<a href="https://colab.research.google.com/github/alimor37/WAAM_Weld_ML/blob/main/Phase4_MinMacScaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.model_selection import LeaveOneOut, GridSearchCV, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')

# 1. Load the data directly from GitHub Raw URL
github_url = "https://raw.githubusercontent.com/alimor37/WAAM_Weld_ML/main/Cleaned_data.xlsx"

print("📥 Loading data from GitHub...")
df = pd.read_excel(github_url)
print("✅ Data loaded successfully!\n")

# Set exact target columns
target_rp = 'Rp0.2_Mpa'
target_rm = 'Rm_Mpa'

# 2. Data Cleaning
df[target_rp] = pd.to_numeric(df[target_rp], errors='coerce')
df[target_rm] = pd.to_numeric(df[target_rm], errors='coerce')

if df['t85_s'].dtype == object:
    df['t85_s'] = df['t85_s'].astype(str).apply(
        lambda x: sum([float(i) for i in x.split('-')])/2 if '-' in x
        else float(x) if x.replace('.', '', 1).isdigit() else None
    )

trace_elements = ['Nb', 'B', 'O', 'Ti', 'Al', 'S', 'P']
for col in trace_elements:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('<', '').str.strip(), errors='coerce')

if 'SinglePass' not in df.columns:
    if 'Weld_Method' in df.columns:
        df['SinglePass'] = ((df['No_of_Beads'] == 1) | (df['Weld_Method'].astype(str).str.strip().isin(['Rapid Arc', 'Laser-Hybrid']))).astype(int)
    else:
        df['SinglePass'] = (df['No_of_Beads'] == 1).astype(int)

df['Base1100'] = df['Base_Material'].astype(str).apply(lambda x: 1 if '1100' in str(x) else 0)

# 3. Define Features
chemistry_features = ['C', 'Si', 'Mn', 'P', 'S', 'Cr', 'Ni', 'Mo', 'V', 'Nb', 'Cu', 'Al', 'Ti', 'B', 'O', 'N']
available_chem = [col for col in chemistry_features if col in df.columns]
process_features = ['Pcm', 't85_s', 'No_of_Beads', 'Dilution_Ni_pct', 'SinglePass', 'Base1100']
features_full = process_features + available_chem

loo = LeaveOneOut()

# =====================================================================
# FUNCTION FOR HYPERPARAMETER TUNING
# =====================================================================
def tune_model(target_col, model_name, model, param_grid):
    print(f"\n{'='*65}")
    print(f"🚀 EXTENDED TUNING {model_name} FOR: {target_col}")
    print(f"🔄 Scaling Method: MinMaxScaler")
    print(f"{'='*65}")

    valid_df = df.dropna(subset=process_features + [target_col]).copy()
    X = valid_df[features_full].values
    y = valid_df[target_col].values

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', MinMaxScaler()),
        ('regressor', model)
    ])

    grid_search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=loo,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X, y)
    best_pipe = grid_search.best_estimator_

    print(f"\n✅ BEST PARAMETERS FOUND:")
    for param, value in grid_search.best_params_.items():
        print(f"   - {param.replace('regressor__', '')}: {value}")

    y_pred = cross_val_predict(best_pipe, X, y, cv=loo)
    best_r2 = r2_score(y, y_pred)
    best_rmse = np.sqrt(mean_squared_error(y, y_pred))

    print(f"\n🏆 OPTIMIZED RESULTS:")
    print(f"   - Optimized LOOCV R²:   {best_r2:.3f}")
    print(f"   - Optimized LOOCV RMSE: {best_rmse:.1f} MPa")
    print("="*65)

# =====================================================================
# 4. RUN TUNING
# =====================================================================
svr_grid = {
    'regressor__kernel': ['rbf', 'linear', 'poly'],
    'regressor__degree': [2, 3],
    'regressor__C': [10, 100, 500, 1000, 2000, 5000],
    'regressor__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'regressor__epsilon': [0.1, 1, 5, 10, 20]
}
tune_model(target_rp, 'SVR', SVR(), svr_grid)

en_grid = {
    'regressor__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1, 5, 10],
    'regressor__l1_ratio': [0.01, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0],
    'regressor__fit_intercept': [True, False],
    'regressor__tol': [1e-4, 1e-3, 1e-2]
}
tune_model(target_rm, 'Elastic Net', ElasticNet(max_iter=20000), en_grid)

📥 Loading data from GitHub...
✅ Data loaded successfully!


🚀 EXTENDED TUNING SVR FOR: Rp0.2_Mpa
🔄 Scaling Method: MinMaxScaler
Fitting 28 folds for each of 900 candidates, totalling 25200 fits

✅ BEST PARAMETERS FOUND:
   - C: 500
   - degree: 2
   - epsilon: 5
   - gamma: auto
   - kernel: rbf

🏆 OPTIMIZED RESULTS:
   - Optimized LOOCV R²:   0.791
   - Optimized LOOCV RMSE: 49.2 MPa

🚀 EXTENDED TUNING Elastic Net FOR: Rm_Mpa
🔄 Scaling Method: MinMaxScaler
Fitting 28 folds for each of 384 candidates, totalling 10752 fits

✅ BEST PARAMETERS FOUND:
   - alpha: 0.05
   - fit_intercept: True
   - l1_ratio: 0.5
   - tol: 0.01

🏆 OPTIMIZED RESULTS:
   - Optimized LOOCV R²:   0.743
   - Optimized LOOCV RMSE: 39.0 MPa
